# 03 - QLoRA Fine-Tuning Tutorial (facebook/opt-350m)

This notebook focuses on the QLoRA branch and inspects real 4-bit training/evaluation artifacts.

## QLoRA training metrics (from artifacts)

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent.resolve()
ART = ROOT / 'artifacts'
MODELS = ROOT / 'models'

qlora_train = json.loads((ART / 'metrics' / 'qlora_train_metrics.json').read_text())
qlora_train

{'train': {'train_runtime': 51.3507,
  'train_samples_per_second': 1.869,
  'train_steps_per_second': 0.234,
  'total_flos': 10462867095552.0,
  'train_loss': 3.627966602643331,
  'epoch': 0.05333333333333334},
 'eval': {'eval_loss': 2.897284746170044,
  'eval_runtime': 12.2634,
  'eval_samples_per_second': 24.463,
  'eval_steps_per_second': 12.231,
  'epoch': 0.05333333333333334},
 'model': 'facebook/opt-350m',
 'method': 'qlora',
 'train_samples': 1800,
 'validation_samples': 300}

## Compare QLoRA baseline vs tuned performance

In [2]:
metrics = pd.read_csv(ART / 'metrics' / 'evaluation_metrics.csv')
metrics[metrics['run_name'].isin(['baseline_qlora','tuned_qlora'])]

,run_name,model_name,quantized_4bit,adapter_path,n_samples,accuracy,macro_f1
2,baseline_qlora,facebook/opt-350m,True,NaN,40,0.1000,0.136508
3,tuned_qlora,facebook/opt-350m,True,/home/ahmad/AI/Projects/lora-qlora-finetuning-...,80,0.3625,0.140478


## Inspect tuned QLoRA predictions

In [3]:
preds = pd.read_csv(ART / 'tables' / 'predictions.csv')
preds[preds['run_name']=='tuned_qlora'].head(20)

,run_name,text,gold_label,raw_completion,pred_label
160,tuned_qlora,i was feeling really troubled and down over wh...,sadness,"sadness, joy,",sadness
161,tuned_qlora,i feel so thrilled to have three such distingu...,joy,"joy, joy,",joy
162,tuned_qlora,i feel is that the most likeable characters ar...,joy,"sadness, joy,",sadness
163,tuned_qlora,i tune out the rest of the world and focus on ...,joy,"sadness, joy,",sadness
164,tuned_qlora,i sit here writing this i feel unhappy inside,sadness,"sadness, joy,",sadness
165,tuned_qlora,im feeling and if ive liked being pregnant,love,"sadness, joy,",sadness
166,tuned_qlora,im very hurt and i feel unimportant,sadness,"sadness, joy,",sadness
167,tuned_qlora,i used to be able to hang around talk with the...,anger,"sadness, joy,",sadness
168,tuned_qlora,i don t have the feeling of divine vibrations,joy,"sadness, joy,",sadness
169,tuned_qlora,i vented my feelings towards the pathetic excu...,sadness,"anger, fear,",anger


## QLoRA adapter path and files

In [4]:
qlora_adapter = MODELS / 'qlora_opt350m' / 'adapter'
print(qlora_adapter)
for p in sorted(qlora_adapter.glob('*')):
    print(p.name)

/home/ahmad/AI/Projects/lora-qlora-finetuning-lab/models/qlora_opt350m/adapter
README.md
adapter_config.json
adapter_model.safetensors
tokenizer.json
tokenizer_config.json
training_args.bin
